In [3]:
!pip install tokenizers -q

import os
from tokenizers import ByteLevelBPETokenizer

# Verify the splits are accessible
data_dir = "/kaggle/input/datasets/ibrahimhany202200518/gec-c4-5m-splits/data"
for f in os.listdir(data_dir):
    size_mb = os.path.getsize(os.path.join(data_dir, f)) / 1e6
    print(f"  {f} → {size_mb:.1f} MB")

  test.tsv → 21.2 MB
  train.tsv → 1184.8 MB
  val.tsv → 41.3 MB


## 1. Sample 500k sentences for tokenizer training

We sample from the training split only (not val/test).
We combine both input and output sentences so the tokenizer
sees the full vocabulary of both corrupted and clean text.

In [7]:
import random

SEED         = 42
SAMPLE_SIZE  = 500_000
TRAIN_PATH   = "/kaggle/input/datasets/ibrahimhany202200518/gec-c4-5m-splits/data/train.tsv"

random.seed(SEED)

sentences = []
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            sentences.append(parts[0])  # input
            sentences.append(parts[1])  # output

# Sample 500k from the combined pool
random.shuffle(sentences)
sentences = sentences[:SAMPLE_SIZE]

print(f"Total sentences sampled : {len(sentences):,}")
print(f"Sample preview:")
for s in sentences[:3]:
    print(f"  {s[:80]}")

Total sentences sampled : 500,000
Sample preview:
  So, what could be bad about that?
  In the meantime, we're rolling with it and doing what needs to be done.
  Best feeling and comfortable sleep.


## 2. Save sentences to a temporary text file

The tokenizer trainer expects a plain text file — one sentence per line.

In [8]:
import time

TEMP_PATH = "/kaggle/working/tokenizer_train_corpus.txt"

start = time.time()
with open(TEMP_PATH, "w", encoding="utf-8") as f:
    for s in sentences:
        f.write(s + "\n")

elapsed = time.time() - start
size_mb = os.path.getsize(TEMP_PATH) / 1e6
print(f"Saved {len(sentences):,} sentences → {size_mb:.1f} MB ({elapsed:.1f}s)")

Saved 500,000 sentences → 62.3 MB (0.4s)


## 3. Train the BPE tokenizer

In [9]:
VOCAB_SIZE   = 16_000
OUT_DIR      = "/kaggle/working/tokenizer"
os.makedirs(OUT_DIR, exist_ok=True)

tokenizer = ByteLevelBPETokenizer()

start = time.time()
tokenizer.train(
    files=[TEMP_PATH],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=["<pad>", "<unk>", "<s>", "</s>"]
)
elapsed = time.time() - start

tokenizer.save_model(OUT_DIR)
print(f"Tokenizer trained in {elapsed:.1f}s")
print(f"Saved to {OUT_DIR}")
print(f"Files: {os.listdir(OUT_DIR)}")




Tokenizer trained in 16.6s
Saved to /kaggle/working/tokenizer
Files: ['vocab.json', 'merges.txt']


## 4. Sanity check — encode & decode round trip

In [10]:
from tokenizers import ByteLevelBPETokenizer

# Reload from saved files
tokenizer = ByteLevelBPETokenizer(
    os.path.join(OUT_DIR, "vocab.json"),
    os.path.join(OUT_DIR, "merges.txt"),
)
tokenizer.add_special_tokens(["<pad>", "<unk>", "<s>", "</s>"])

# Test sentences
test_sentences = [
    "Bitcoin is for $7,094 this morning, which CoinDesk says.",
    "She go to the store yesterday.",
    "The new 15 percents tax bracket kicks in.",
]

print("Round-trip encode → decode test:\n")
for s in test_sentences:
    encoded = tokenizer.encode(s)
    decoded = tokenizer.decode(encoded.ids)
    print(f"  Original : {s}")
    print(f"  Tokens   : {encoded.tokens[:10]} ...")
    print(f"  IDs      : {encoded.ids[:10]} ...")
    print(f"  Decoded  : {decoded}")
    print()

print(f"Vocab size : {tokenizer.get_vocab_size():,}")
print(f"PAD id     : {tokenizer.token_to_id('<pad>')}")
print(f"UNK id     : {tokenizer.token_to_id('<unk>')}")
print(f"BOS id     : {tokenizer.token_to_id('<s>')}")
print(f"EOS id     : {tokenizer.token_to_id('</s>')}")

Round-trip encode → decode test:

  Original : Bitcoin is for $7,094 this morning, which CoinDesk says.
  Tokens   : ['B', 'it', 'coin', 'Ġis', 'Ġfor', 'Ġ$', '7', ',', '09', '4'] ...
  IDs      : [37, 275, 6538, 324, 316, 811, 26, 15, 5441, 23] ...
  Decoded  : Bitcoin is for $7,094 this morning, which CoinDesk says.

  Original : She go to the store yesterday.
  Tokens   : ['She', 'Ġgo', 'Ġto', 'Ġthe', 'Ġstore', 'Ġyesterday', '.'] ...
  IDs      : [2992, 609, 288, 268, 2043, 6041, 17] ...
  Decoded  : She go to the store yesterday.

  Original : The new 15 percents tax bracket kicks in.
  Tokens   : ['The', 'Ġnew', 'Ġ15', 'Ġper', 'c', 'ents', 'Ġtax', 'Ġbra', 'cket', 'Ġk'] ...
  IDs      : [404, 580, 1499, 592, 70, 610, 1912, 2586, 7592, 500] ...
  Decoded  : The new 15 percents tax bracket kicks in.

Vocab size : 16,000
PAD id     : 0
UNK id     : 1
BOS id     : 2
EOS id     : 3
